# NB08 — Leave-one-center-out (cross-center generalization)

The paper's per-center analysis is *post-hoc* (train and test pooled, then stratified
by `LOCAL`). This notebook runs the real test of transfer: **train on one center,
test on the other**, both directions.

- **A→B**: train on LOCAL=1 (Center A, 720 imgs), test on all of LOCAL=2 (Center B, 1,270).
- **B→A**: train on LOCAL=2, test on all of LOCAL=1.

Model and recipe match the paper's best config (Swin-Tiny + CCR, λ=0.6). The
co-occurrence matrix `C` is estimated **only on the training center** (no leakage from
the test center). Thresholds are tuned on an internal validation hold-out of the
training center and then frozen for the test center.

## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [2]:
import json, random, time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score, roc_auc_score)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens...')
        !unzip -q /content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/anonymous-V3/Trilha3-V2')
    IMGS_DIR = Path('e:/anonymous-V3/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'nb08'
CKPT_DIR    = BASE_DIR / 'checkpoints'
FIGS_DIR    = BASE_DIR / 'figs' / 'training'
for d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_CLASSES   = len(CORE_LABELS)

IMG_SIZE     = 224
BATCH_SIZE   = 128          # G4 96 GB: pode subir; ambas direções usam o mesmo batch
MAX_EPOCHS   = 50
LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
ES_PATIENCE  = 10
LR_PATIENCE  = 4
LR_FACTOR    = 0.5
LAMBDA       = 0.6          # CCR — mesmo do melhor modelo
VAL_FRAC     = 0.15         # hold-out interno do centro de treino p/ tunar threshold
SEED         = 42
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Direções: (nome, LOCAL_treino, LOCAL_teste)
DIRECTIONS = [
    {'id': 'A_to_B', 'train_local': 1, 'test_local': 2},
    {'id': 'B_to_A', 'train_local': 2, 'test_local': 1},
]

print(f'Device   : {DEVICE}')
print(f'λ (CCR)  : {LAMBDA}')
print(f'Direções : {[d["id"] for d in DIRECTIONS]}')

## 2. Master table with center label

The five `fold_*_test.csv` are disjoint and jointly cover all 1,990 images exactly once
(5-fold CV), so concatenating them yields the full set with the `LOCAL` column.

In [4]:
parts = [pd.read_csv(SPLITS_DIR / f'fold_{k}_test.csv') for k in range(5)]
master = pd.concat(parts, ignore_index=True).drop_duplicates('image_name').reset_index(drop=True)

assert 'LOCAL' in master.columns, 'Coluna LOCAL ausente!'
print(f'Master: {len(master)} imagens únicas (esperado ~1990).')
for loc in sorted(master['LOCAL'].unique()):
    sub = master[master['LOCAL'] == loc]
    prev = {l: int(sub[l].sum()) for l in CORE_LABELS}
    print(f'  LOCAL={loc}: {len(sub)} imagens | positivos por classe: {prev}')

## 3. Dataset, model, loss, metrics (paper recipe)

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
def get_transforms(split):
    normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        return T.Compose([
            T.Resize((int(IMG_SIZE*1.15), int(IMG_SIZE*1.15))),
            T.RandomHorizontalFlip(p=0.5), T.RandomRotation(degrees=10),
            T.RandomResizedCrop(size=IMG_SIZE, scale=(0.85,1.0), ratio=(0.9,1.1)),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.ToTensor(), normalize])
    return T.Compose([
        T.Resize((int(IMG_SIZE*1.14), int(IMG_SIZE*1.14))),
        T.CenterCrop(IMG_SIZE), T.ToTensor(), normalize])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df = df.reset_index(drop=True); self.imgs_dir = imgs_dir; self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform: img = self.transform(img)
        return img, labels

def compute_pos_weights(df):
    pos = df[CORE_LABELS].sum(axis=0).values
    neg = len(df) - pos
    return torch.tensor(neg/(pos+1e-6), dtype=torch.float32).to(DEVICE)

def compute_cooccurrence_matrix(df):
    labels = df[CORE_LABELS].values.astype(float)
    C = np.zeros((N_CLASSES, N_CLASSES))
    for i in range(N_CLASSES):
        mask_i = labels[:, i] == 1; n_i = mask_i.sum()
        if n_i == 0: continue
        for j in range(N_CLASSES):
            if i == j: continue
            C[i][j] = labels[mask_i, j].sum() / n_i
    return torch.tensor(C, dtype=torch.float32).to(DEVICE)

class M2CoocLoss(nn.Module):
    def __init__(self, pos_weight, cooc_matrix, lam):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.C = cooc_matrix; self.lam = lam
    def forward(self, logits, targets):
        loss_bce = self.bce(logits, targets)
        if self.lam == 0.0: return loss_bce
        probs = torch.sigmoid(logits)
        diff = torch.relu(probs.unsqueeze(2) - probs.unsqueeze(1))
        return loss_bce + self.lam * (self.C.unsqueeze(0) * diff.pow(2)).mean()

def build_swin_tiny(n_classes=N_CLASSES):
    return timm.create_model('swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
                             pretrained=True, num_classes=n_classes)
def build_optimizer(model):
    head = list(model.head.parameters())
    back = [p for n,p in model.named_parameters() if 'head' not in n]
    return torch.optim.AdamW([{'params': back, 'lr': LR_BACKBONE},
                              {'params': head, 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)

def optimize_thresholds(probs, labels):
    thr = np.zeros(N_CLASSES)
    for i in range(N_CLASSES):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            if preds.sum() == 0: continue
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        thr[i] = best_t
    return thr

def compute_metrics(probs, labels, thresholds=None):
    if thresholds is None: thresholds = np.full(N_CLASSES, 0.5)
    preds = (probs >= thresholds).astype(int)
    f1_per = f1_score(labels, preds, average=None, zero_division=0)
    pr_per = precision_score(labels, preds, average=None, zero_division=0)
    rc_per = recall_score(labels, preds, average=None, zero_division=0)
    auprc, rocauc = [], []
    for i in range(N_CLASSES):
        auprc.append(average_precision_score(labels[:,i], probs[:,i]) if labels[:,i].sum()>0 else float('nan'))
        rocauc.append(roc_auc_score(labels[:,i], probs[:,i]) if 0<labels[:,i].sum()<len(labels) else float('nan'))
    result = {'f1_macro': float(np.nanmean(f1_per)),
              'f1_micro': float(f1_score(labels, preds, average='micro', zero_division=0)),
              'pr_auc_macro': float(np.nanmean(auprc)),
              'roc_auc_macro': float(np.nanmean(rocauc))}
    for i, label in enumerate(CORE_LABELS):
        result[f'f1_{label}']    = float(f1_per[i])
        result[f'prec_{label}']  = float(pr_per[i])
        result[f'rec_{label}']   = float(rc_per[i])
        result[f'auprc_{label}'] = float(auprc[i])
        result[f'thr_{label}']   = float(thresholds[i])
    return result

print('Componentes (receita do paper) definidos.')

## 4. Training loop — one direction

In [6]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); probs, labels = [], []
    for imgs, lab in loader:
        probs.append(torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy())
        labels.append(lab.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def split_train_val(df, val_frac, seed):
    """Hold-out interno aleatório (seeded) do centro de treino p/ tunar threshold."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    n_val = int(len(df) * val_frac)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]
    return df.iloc[tr_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)

def train_direction(direction, save_checkpoint=True, verbose=True):
    set_seed(SEED)
    df_train_center = master[master['LOCAL'] == direction['train_local']].reset_index(drop=True)
    df_test         = master[master['LOCAL'] == direction['test_local']].reset_index(drop=True)
    df_tr, df_val   = split_train_val(df_train_center, VAL_FRAC, SEED)

    if verbose:
        print(f'  Treino (LOCAL={direction["train_local"]}): {len(df_tr)} | '
              f'val: {len(df_val)} | teste (LOCAL={direction["test_local"]}): {len(df_test)}')

    ds_tr  = GastroscopyDataset(df_tr,  IMGS_DIR, get_transforms('train'))
    ds_val = GastroscopyDataset(df_val, IMGS_DIR, get_transforms('val'))
    ds_te  = GastroscopyDataset(df_test,IMGS_DIR, get_transforms('val'))
    dl_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    dl_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model     = build_swin_tiny().to(DEVICE)
    pos_w     = compute_pos_weights(df_tr)
    cooc_mat  = compute_cooccurrence_matrix(df_tr)   # só centro de treino
    criterion = M2CoocLoss(pos_w, cooc_mat, LAMBDA)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                          patience=LR_PATIENCE, factor=LR_FACTOR)
    best_val_f1, best_weights, es = -1.0, None, 0
    history = {'train_loss': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); running = 0.0
        for imgs, lab in dl_tr:
            imgs, lab = imgs.to(DEVICE), lab.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lab)
            loss.backward(); optimizer.step()
            running += loss.item() * len(imgs)
        train_loss = running / len(ds_tr)
        vp, vl = evaluate(model, dl_val)
        vthr = optimize_thresholds(vp, vl)
        vf1  = compute_metrics(vp, vl, vthr)['f1_macro']
        scheduler.step(vf1)
        history['train_loss'].append(train_loss); history['val_f1'].append(vf1)
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'    Epoch {epoch:>3} | loss={train_loss:.4f} | val_F1={vf1:.4f}')
        if vf1 > best_val_f1 + 1e-4:
            best_val_f1, best_weights, es = vf1, deepcopy(model.state_dict()), 0
        else:
            es += 1
            if es >= ES_PATIENCE:
                if verbose: print(f'    Early stopping na época {epoch}.')
                break

    model.load_state_dict(best_weights)
    if save_checkpoint:
        torch.save(best_weights, CKPT_DIR / f'swin_loco_{direction["id"]}.pt')
    tp, tl = evaluate(model, dl_te)
    test_met = compute_metrics(tp, tl, vthr)
    test_met.update({'direction': direction['id'],
                     'train_local': direction['train_local'], 'test_local': direction['test_local'],
                     'n_train': len(df_tr), 'n_val': len(df_val), 'n_test': len(df_test),
                     'best_val_f1': best_val_f1, 'epochs_trained': len(history['train_loss']),
                     'val_thresholds': vthr.tolist(), 'history': history})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return test_met

print('Loop de treino (por direção) definido.')

## 5. Execution (with resume)

In [7]:
RUN_FAST_CHECK = False
if RUN_FAST_CHECK:
    MAX_EPOCHS = 5
    print('MODO FAST CHECK: 5 épocas por direção.')

results_file = RESULTS_DIR / 'loco_results.json'
if results_file.exists():
    print('>> ENCONTRADO RESULTADO ANTERIOR! Retomando... <<')
    with open(results_file, encoding='utf-8') as f:
        all_results = json.load(f)
else:
    all_results = {}

for direction in DIRECTIONS:
    did = direction['id']
    if did in all_results:
        print(f'\n[PULANDO] Direção {did} já concluída.')
        continue
    print(f'\n{"="*60}\nDireção: {did}  (treina LOCAL={direction["train_local"]} → '
          f'testa LOCAL={direction["test_local"]})\n{"="*60}')
    t0 = time.time()
    met = train_direction(direction, save_checkpoint=not RUN_FAST_CHECK)
    met['elapsed_sec'] = round(time.time() - t0, 1)
    all_results[did] = met
    print(f'\n  → Cross-center Macro-F1={met["f1_macro"]:.4f}  PR-AUC={met["pr_auc_macro"]:.4f}')
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print('  Salvo.')

## 6. Summary, comparison vs. post-hoc, and figures (EN + PT)

We contrast the true cross-center F1 (this notebook) with the paper's *post-hoc*
per-center numbers (pooled training, stratified evaluation: A=0.506, B=0.663). A drop
from post-hoc to true LOCO quantifies the optimism of the pooled estimate.

In [8]:
POSTHOC = {1: 0.506, 2: 0.663}  # F1 por centro no paper (pooled, post-hoc)

rows = []
for did, r in all_results.items():
    rows.append({
        'Direction': did,
        'Train→Test': f'LOCAL{r["train_local"]}→LOCAL{r["test_local"]}',
        'n_test': r['n_test'],
        'F1_cross': round(r['f1_macro'], 3),
        'PR_AUC': round(r['pr_auc_macro'], 3),
        'F1_posthoc_test_center': POSTHOC.get(r['test_local'], float('nan')),
        'drop_vs_posthoc': round(r['f1_macro'] - POSTHOC.get(r['test_local'], float('nan')), 3),
    })
df_sum = pd.DataFrame(rows).set_index('Direction')
print('=== LEAVE-ONE-CENTER-OUT ===')
print(df_sum.to_string())
df_sum.to_csv(RESULTS_DIR / 'tabela_nb08.csv')

TXT = {'en': ('Leave-one-center-out: per-class F1', 'Macro-F1', 'Class'),
       'pt': ('Leave-one-center-out: F1 por classe', 'Macro-F1', 'Classe')}

def make_fig(lang):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    dirs = list(all_results.keys())
    macro = [all_results[d]['f1_macro'] for d in dirs]
    axes[0].bar(dirs, macro, color=['#1565C0', '#2E7D32'], edgecolor='white')
    for i, m in enumerate(macro):
        axes[0].text(i, m+0.005, f'{m:.3f}', ha='center', va='bottom', fontsize=10)
    axes[0].set_ylabel(TXT[lang][1]); axes[0].set_ylim(0, max(macro)+0.1)
    axes[0].set_title(TXT[lang][0] if lang=='en' else TXT[lang][0], fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    heat = {d: [all_results[d].get(f'f1_{l}', float('nan')) for l in CORE_LABELS] for d in dirs}
    df_heat = pd.DataFrame(heat, index=CORE_LABELS).T
    sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.0, vmax=0.85,
                linewidths=0.4, ax=axes[1], cbar_kws={'label': 'F1'})
    axes[1].set_title(TXT[lang][0], fontweight='bold')
    plt.tight_layout()
    out = FIGS_DIR / f'nb08_loco_{lang}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Figura salva: {out.name}')

for lang in ['en', 'pt']:
    make_fig(lang)